### 08 - Inverse PINN Calibration on Real NDX Option Data

We repeat the inverse PINN experiment, but replace the synthetic Black-Scholes data with real NDX European call option market data.

r and sigma are genuinely unknown here — there is no ground truth to compare
against directly. Instead we train four separate PINNs using 50, 100, 150,
and 200 randomly selected real observations for the data-fit term, and
compare the resulting (r, sigma) estimates against each other and, later,
against a regression baseline.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from inverse_pinn_model import InversePINN
import inverse_pinn_training as ipt
from inverse_pinn_training import black_scholes_residual

device = torch.device("cpu")
dtype = torch.float64

In [3]:
data_path = Path("NDX_call_data.csv")

raw_df = pd.read_csv(data_path, index_col=0)

assert raw_df["strike_price"].nunique() == 1, "expected a single strike"
K = float(raw_df["strike_price"].iloc[0])

print("Raw shape:", raw_df.shape)
print("K:", K)
raw_df.head()

Raw shape: (239, 7)
K: 15700.0


,spot_price,current_time,strike_price,best_bid,best_offer,ticker,market_price
0,14137.230469,0.989041,15700.0,493.5,528.5,NDX,511.00
1,14270.419922,0.986301,15700.0,530.6,557.9,NDX,544.25
2,14274.240234,0.983562,15700.0,515.7,544.3,NDX,530.00
3,14365.959961,0.980822,15700.0,552.5,592.4,NDX,572.45
4,14345.179688,0.978082,15700.0,539.3,575.6,NDX,557.45


In [ ]:
# current_time in the csv is time-to-maturity (tau), decreasing toward 0
# convert t = T - tau

T = float(raw_df["current_time"].max())

t_min = 0.0
t_max = T

S_min = 0.0
S_max = float(raw_df["spot_price"].max())

print("K:", K)
print("T:", T)
print("S_max:", S_max)

K: 15700.0
T: 0.989041095890411
S_max: 16573.33984375


In [ ]:
# Change K, and T in synthetic data

ipt.K = K
ipt.T = T

compute_losses = ipt.compute_losses

print("inverse_pinn_training.K is now:", ipt.K)
print("inverse_pinn_training.T is now:", ipt.T)

inverse_pinn_training.K is now: 15700.0
inverse_pinn_training.T is now: 0.989041095890411


In [ ]:
# rearrange original csv 
real_df = pd.DataFrame({
    "t": T - raw_df["current_time"],
    "S": raw_df["spot_price"],
    "V": raw_df["market_price"],
})

real_df = real_df.sort_values("t").reset_index(drop=True)

print("real_df shape:", real_df.shape)
real_df.head()

real_df shape: (239, 3)


,t,S,V
0,0.000000,14137.230469,511.00
1,0.002740,14270.419922,544.25
2,0.005479,14274.240234,530.00
3,0.008219,14365.959961,572.45
4,0.010959,14345.179688,557.45


In [7]:
# convert all real observations to tensors
t_all = torch.tensor(
    real_df["t"].to_numpy(), dtype=dtype, device=device
).reshape(-1, 1)

S_all = torch.tensor(
    real_df["S"].to_numpy(), dtype=dtype, device=device
).reshape(-1, 1)

V_all = torch.tensor(
    real_df["V"].to_numpy(), dtype=dtype, device=device
).reshape(-1, 1)

u_all = V_all / K

print("t_all:", t_all.shape)
print("S_all:", S_all.shape)
print("u_all:", u_all.shape)

t_all: torch.Size([239, 1])
S_all: torch.Size([239, 1])
u_all: torch.Size([239, 1])


In [8]:
data_sizes = [50, 100, 150, 200]

def sample_data_subset(n):
    """
    Randomly sample n observations (without replacement) from the real data pool
    Each call draws a fresh, independent random subset
    """
    n_pool = t_all.shape[0]
    idx = torch.randperm(n_pool)[:n]
    return t_all[idx], S_all[idx], u_all[idx]

In [11]:
# The same 500 pde collocation points for each [50, 100, 150, 200]

n_pde = 500

t_f = torch.rand(n_pde, 1, dtype=dtype, device=device) * (t_max - t_min) + t_min
S_f = torch.rand(n_pde, 1, dtype=dtype, device=device) * (S_max - S_min) + S_min

print("t_f:", t_f.shape, "range:", t_f.min().item(), t_f.max().item())
print("S_f:", S_f.shape, "range:", S_f.min().item(), S_f.max().item())

t_f: torch.Size([500, 1]) range: 0.0005035376795703136 0.9856614848819177
S_f: torch.Size([500, 1]) range: 19.753928207559255 16547.23452023355


In [12]:
# 500 terminal rand points
n_terminal = 500

t_terminal = torch.full((n_terminal, 1), t_max, dtype=dtype, device=device)
S_terminal = torch.rand(n_terminal, 1, dtype=dtype, device=device) * (S_max - S_min) + S_min

print("t_terminal:", t_terminal.shape, "value:", t_terminal.unique().item())
print("S_terminal:", S_terminal.shape, "range:", S_terminal.min().item(), S_terminal.max().item())

t_terminal: torch.Size([500, 1]) value: 0.989041095890411
S_terminal: torch.Size([500, 1]) range: 82.80893150476189 16562.858153505687


In [14]:
# boundary points (left = 500, right = 500)
n_boundary = 500

t_boundary = torch.rand(n_boundary, 1, dtype=dtype, device=device) * (t_max - t_min) + t_min
S_boundary_left = torch.full((n_boundary, 1), S_min, dtype=dtype, device=device)
S_boundary_right = torch.full((n_boundary, 1), S_max, dtype=dtype, device=device)

print("t_boundary:", t_boundary.shape, "range:", t_boundary.min().item(), t_boundary.max().item())
print("S_boundary_left value:", S_boundary_left.unique().item())
print("S_boundary_right value:", S_boundary_right.unique().item())

t_boundary: torch.Size([500, 1]) range: 0.0021218341322113356 0.9884011085031709
S_boundary_left value: 0.0
S_boundary_right value: 16573.33984375


In [15]:
def build_model(seed):
    torch.manual_seed(seed)
    model = InversePINN(
        t_min=t_min,
        t_max=t_max,
        S_min=S_min,
        S_max=S_max,
        hidden_width=64,
        hidden_layers=4,
        r_initial=0.10,
        sigma_initial=0.40,
    ).to(device=device, dtype=dtype)
    return model

In [16]:
learning_rate = 1e-3
num_epochs = 30000
print_every = 1000

In [21]:
results = {}

for n_data in data_sizes:

    print(f"\nTraining with n_data = {n_data}")

    t_train, S_train, u_train = sample_data_subset(n_data)

    model = build_model(seed=n_data)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    history = {
        "epoch": [],
        "total_loss": [],
        "data_loss": [],
        "pde_loss": [],
        "terminal_loss": [],
        "boundary_loss": [],
        "r_estimate": [],
        "sigma_estimate": [],
    }

    for epoch in range(1, num_epochs + 1):

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        losses = compute_losses(
            model=model,
            t_train=t_train,
            S_train=S_train,
            u_train=u_train,
            t_f=t_f,
            S_f=S_f,
            t_terminal=t_terminal,
            S_terminal=S_terminal,
            t_boundary=t_boundary,
            S_boundary_left=S_boundary_left,
            S_boundary_right=S_boundary_right,
        )

        losses["total"].backward()

        optimizer.step()


        history["epoch"].append(
            epoch
        )

        history["total_loss"].append(
            losses["total"].item()
        )

        history["data_loss"].append(
            losses["data"].item()
        )

        history["pde_loss"].append(
            losses["pde"].item()
        )

        history["terminal_loss"].append(
            losses["terminal"].item()
        )

        history["boundary_loss"].append(
            losses["boundary"].item()
        )

        history["r_estimate"].append(
            model.r.item()
        )

        history["sigma_estimate"].append(
            model.sigma.item()
        )

        if epoch == 1 or epoch % print_every == 0:

            print(
                f"Epoch {epoch:6d} | "
                f"Total: {losses['total'].item():.6e} | "
                f"Data: {losses['data'].item():.6e} | "
                f"PDE: {losses['pde'].item():.6e} | "
                f"r: {model.r.item():.6f} | "
                f"sigma: {model.sigma.item():.6f}"
            )



    # Evaluate final PINN on ALL 239 real market observations

    model.eval()

    with torch.no_grad():

        # PINN outputs normalized price:
        # u = V / K
        u_all_prediction = model(
            t_all,
            S_all
        )

        # Convert back to original option-price scale
        V_all_prediction = (
            K * u_all_prediction
        )

        # Pricing errors on all 239 real observations
        full_data_error = (
            V_all_prediction
            -
            V_all
        )

        # MSE
        pinn_full_mse = torch.mean(
            full_data_error ** 2
        ).item()

        # RMSE
        pinn_full_rmse = np.sqrt(
            pinn_full_mse
        )

        # MAE
        pinn_full_mae = torch.mean(
            torch.abs(
                full_data_error
            )
        ).item()


    print("\nFinal evaluation on ALL real observations:")

    print(
        f"r_final         = {model.r.item():.10f}"
    )

    print(
        f"sigma_final     = {model.sigma.item():.10f}"
    )

    print(
        f"Full-data MSE   = {pinn_full_mse:.10f}"
    )

    print(
        f"Full-data RMSE  = {pinn_full_rmse:.10f}"
    )

    print(
        f"Full-data MAE   = {pinn_full_mae:.10f}"
    )



    results[n_data] = {
        "model": model,
        "history": history,
        "r_final": model.r.item(),
        "sigma_final": model.sigma.item(),
        "full_data_mse": pinn_full_mse,
        "full_data_rmse": pinn_full_rmse,
        "full_data_mae": pinn_full_mae,
    }

    torch.save(
        {
            "n_data": n_data,

            "model_state_dict": model.state_dict(),

            "r_final": model.r.item(),
            "sigma_final": model.sigma.item(),

            "full_data_mse": pinn_full_mse,
            "full_data_rmse": pinn_full_rmse,
            "full_data_mae": pinn_full_mae,

            "K": K,
            "T": T,

            "t_min": t_min,
            "t_max": t_max,

            "S_min": S_min,
            "S_max": S_max,

            "hidden_width": 64,
            "hidden_layers": 4,

            "learning_rate": learning_rate,
            "num_epochs": num_epochs,

            "history": history,
        },

        f"ndx_real_pinn_same_lr_1e-3_30000epochs_n{n_data}_run1.pt"
    )

    print(
        f"Checkpoint saved for n_data = {n_data}"
    )


Training with n_data = 50
Epoch      1 | Total: 7.657330e-01 | Data: 5.697367e-02 | PDE: 1.276622e-01 | r: 0.100095 | sigma: 0.399670
Epoch   1000 | Total: 8.799712e-04 | Data: 8.961388e-05 | PDE: 7.758965e-05 | r: 0.069647 | sigma: 0.311008
Epoch   2000 | Total: 2.855179e-03 | Data: 2.148741e-04 | PDE: 5.989131e-04 | r: 0.059592 | sigma: 0.235149
Epoch   3000 | Total: 3.548519e-04 | Data: 3.795945e-05 | PDE: 5.148478e-05 | r: 0.061104 | sigma: 0.190722
Epoch   4000 | Total: 1.659980e-04 | Data: 1.640055e-05 | PDE: 1.805825e-05 | r: 0.074418 | sigma: 0.147784
Epoch   5000 | Total: 1.397493e-04 | Data: 1.465289e-05 | PDE: 1.207886e-05 | r: 0.083546 | sigma: 0.125169
Epoch   6000 | Total: 1.334486e-04 | Data: 1.413872e-05 | PDE: 1.049138e-05 | r: 0.087315 | sigma: 0.113977
Epoch   7000 | Total: 1.006500e-03 | Data: 1.335424e-04 | PDE: 9.503334e-06 | r: 0.089482 | sigma: 0.109033
Epoch   8000 | Total: 1.290727e-04 | Data: 1.405022e-05 | PDE: 9.928057e-06 | r: 0.090387 | sigma: 0.107464
E

In [22]:
summary_df = pd.DataFrame([
    {
        "n_data": n_data,

        "r_pinn": results[n_data]["r_final"],
        "sigma_pinn": results[n_data]["sigma_final"],

        # Final training losses
        "final_data_loss": results[n_data]["history"]["data_loss"][-1],
        "final_pde_loss": results[n_data]["history"]["pde_loss"][-1],
        "final_terminal_loss": results[n_data]["history"]["terminal_loss"][-1],
        "final_boundary_loss": results[n_data]["history"]["boundary_loss"][-1],
        "final_total_loss": results[n_data]["history"]["total_loss"][-1],

        # Pricing performance on ALL 239 real observations
        "full_data_mse": results[n_data]["full_data_mse"],
        "full_data_rmse": results[n_data]["full_data_rmse"],
        "full_data_mae": results[n_data]["full_data_mae"],
    }

    for n_data in data_sizes
])


summary_df

,n_data,r_pinn,sigma_pinn,final_data_loss,final_pde_loss,final_terminal_loss,final_boundary_loss,final_total_loss,full_data_mse,full_data_rmse,full_data_mae
0,50,0.090337,0.117744,0.000010,0.000015,0.000014,0.000008,0.000104,3022.788290,54.979890,42.121166
1,100,0.090087,0.118733,0.000008,0.000015,0.000016,0.000008,0.000092,2261.888788,47.559319,34.565929
2,150,0.087372,0.117993,0.000012,0.000013,0.000017,0.000008,0.000115,3446.035509,58.702943,43.342012
3,200,0.086219,0.123193,0.000012,0.000017,0.000015,0.000010,0.000122,2866.422519,53.538981,40.500493
